# 🌱 Цифровой двойник — Live сравнение с реальной теплицей

```
ESP32 датчики
    ↓ каждые 15 с
Java Monitoring Server (:8080)
    ↓ REST API
greenhouse_bridge.py        ← этот ноутбук использует его
    ├── fetch_sensor_data()  — получает реальные данные
    ├── build_greenlight_csv() — конвертирует единицы
    ├── run_simulation()     — запускает GreenLight ODE
    └── compare()            — находит отклонения (аномалии)
```

**Как использовать:**
1. Убедитесь, что мониторинговый сервер запущен (`cd monitoringServer && mvn spring-boot:run`)
2. Укажите `DEVICE_ID` и временной период ниже
3. Запустите все ячейки (Shift+Enter)


## ⚙️ 1. Параметры

In [ ]:
import sys
sys.path.insert(0, '..')  # импортируем greenhouse_bridge из корня проекта

from datetime import datetime, timedelta

# ─── Настройте эти параметры ────────────────────────────────────────────────

DEVICE_ID   = 1                       # ID вашего ESP32 устройства в базе
HOURS_BACK  = 24                      # Анализировать последние N часов
API_BASE    = 'http://localhost:8080' # URL мониторингового сервера

# Временной диапазон (None = использовать текущее время)
TO_DT   = None   # datetime(2026, 3, 31, 12, 0) — или None для «сейчас»
FROM_DT = None   # вычисляется автоматически как TO_DT - HOURS_BACK

# ─── Вычисляем диапазон ─────────────────────────────────────────────────────
to_dt   = TO_DT   or datetime.now()
from_dt = FROM_DT or (to_dt - timedelta(hours=HOURS_BACK))

print(f'📡 Устройство:  #{DEVICE_ID}')
print(f'🕐 Период:      {from_dt:%Y-%m-%d %H:%M} — {to_dt:%Y-%m-%d %H:%M}')
print(f'🌐 Сервер:      {API_BASE}')

## 📡 2. Получение данных и запуск симуляции

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from IPython.display import display, clear_output

# Тёмный стиль
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#e6edf3',
    'text.color': '#e6edf3',       'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',      'grid.color': '#21262d',
    'grid.linewidth': 0.8,         'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'legend.labelcolor': '#e6edf3',
    'font.size': 11,               'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

from greenhouse_bridge import run_pipeline, fetch_latest

print('🚀 Запускаем пайплайн...')
print('   Шаг 1/4: получение данных датчиков')
print('   Шаг 2/4: конвертация в формат GreenLight')
print('   Шаг 3/4: запуск симуляции (ODE BDF)')
print('   Шаг 4/4: сравнение и детекция аномалий')
print()

result = run_pipeline(
    device_id=DEVICE_ID,
    hours=HOURS_BACK,
    to_dt=to_dt,
    api_base=API_BASE,
)

sensor_df  = result['sensor_df']
sim_df     = result['sim_df']
compare_df = result['compare_df']

print(f'\n✅ Готово!')
print(f'   Точек датчика:    {len(sensor_df)}')
print(f'   Точек симуляции:  {len(sim_df)}')
print(f'   Точек сравнения:  {len(compare_df)}')
print(f'   Аномалий:         {compare_df["anomaly"].sum()} ({compare_df["anomaly"].mean()*100:.1f}%)')

## 📊 3. Модель vs. Датчики

In [ ]:
cdf = compare_df.copy()
cdf['timestamp'] = pd.to_datetime(cdf['timestamp'])

# Определяем доступные переменные для сравнения
pairs = []
if 'tAir_sim' in cdf.columns and 'tAir_real' in cdf.columns:
    pairs.append(('tAir_sim', 'tAir_real', 'delta_tAir', 'zscore_tAir',
                  '🌡 Температура воздуха', '°C', '#58a6ff', '#ff7b72'))
if 'RH_sim' in cdf.columns and 'RH_real' in cdf.columns:
    pairs.append(('RH_sim', 'RH_real', 'delta_RH', 'zscore_RH',
                  '💧 Влажность воздуха', '%', '#79c0ff', '#f0883e'))
if 'co2_sim' in cdf.columns and 'co2_real' in cdf.columns:
    pairs.append(('co2_sim', 'co2_real', 'delta_co2', 'zscore_co2',
                  '🌬 CO₂', 'ppm', '#3fb950', '#d2a8ff'))

if not pairs:
    print('⚠️  Нет данных для сравнения. Проверьте, что датчик передаёт'
          ' airTemperature, airHumidity, co2Concentration.')
else:
    fig, axes = plt.subplots(len(pairs) * 2, 1,
                              figsize=(16, 4.5 * len(pairs) * 2),
                              sharex=True)
    if len(pairs) * 2 == 2:
        axes = list(axes)

    fig.suptitle(f'📡 Цифровой двойник: Модель (GreenLight) vs. Датчик #{DEVICE_ID}',
                 fontsize=14, y=1.0)

    anom_mask = cdf['anomaly'].values

    ax_idx = 0
    for (sim_col, real_col, delta_col, zscore_col,
         title, unit, color_sim, color_real) in pairs:

        # ── Верхний граф: модель vs реальность ────────────────────────────
        ax = axes[ax_idx]
        ax_idx += 1

        # Зона ±2σ модели
        std_val = cdf[delta_col].std()
        ax.fill_between(cdf['timestamp'],
                        cdf[sim_col] - 2*std_val,
                        cdf[sim_col] + 2*std_val,
                        alpha=0.12, color=color_sim, label='±2σ модели')

        ax.plot(cdf['timestamp'], cdf[sim_col],
                color=color_sim, lw=2.0, label='🔵 Модель GreenLight', zorder=5)
        ax.scatter(cdf['timestamp'], cdf[real_col],
                   color=color_real, s=8, alpha=0.55, label='🔴 Датчик', zorder=4)

        # Аномалии — красные точки поверх
        if anom_mask.any():
            ax.scatter(cdf['timestamp'][anom_mask], cdf[real_col][anom_mask],
                       color='#ff0000', s=40, marker='x', lw=1.5,
                       label='⚠️ Аномалия', zorder=6)

        rmse_val = np.sqrt(np.nanmean((cdf[sim_col] - cdf[real_col])**2))
        ax.set_ylabel(f'{unit}')
        ax.set_title(f'{title}  |  RMSE = {rmse_val:.2f} {unit}')
        ax.legend(loc='upper right', ncol=3, fontsize=8)
        ax.grid(True, alpha=0.35)

        # ── Нижний граф: отклонение + Z-score ─────────────────────────────
        ax = axes[ax_idx]
        ax_idx += 1

        delta = cdf[delta_col]
        ax.fill_between(cdf['timestamp'], delta, alpha=0.35,
                        color=color_sim)
        ax.plot(cdf['timestamp'], delta, color=color_sim, lw=1.2)
        ax.axhline(0, color='#3fb950', lw=1.5)

        # Порог аномалии ±3σ
        thr = 3 * std_val
        ax.axhline(+thr, color='#ff7b72', ls='--', lw=1, alpha=0.7,
                   label=f'+3σ = +{thr:.2f}')
        ax.axhline(-thr, color='#ff7b72', ls='--', lw=1, alpha=0.7,
                   label=f'-3σ = -{thr:.2f}')

        # Закрасить аномальные зоны
        for start, end in _get_anomaly_ranges(cdf['timestamp'].values, anom_mask):
            ax.axvspan(start, end, alpha=0.2, color='#ff7b72')

        ax.set_ylabel(f'Δ{unit} (модель − датчик)')
        ax.set_title('Отклонение от модели')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.35)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m\n%H:%M'))
    axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=max(1, HOURS_BACK//12)))
    axes[-1].set_xlabel('Время')

    plt.tight_layout()
    plt.show()


def _get_anomaly_ranges(timestamps, mask):
    """Возвращает список (start, end) для закрашивания аномальных зон."""
    ranges = []
    in_anomaly = False
    start = None
    for i, (t, m) in enumerate(zip(timestamps, mask)):
        if m and not in_anomaly:
            start = t
            in_anomaly = True
        elif not m and in_anomaly:
            ranges.append((start, timestamps[i-1]))
            in_anomaly = False
    if in_anomaly:
        ranges.append((start, timestamps[-1]))
    return ranges

## ⚠️ 4. Таблица аномалий

In [ ]:
anom_df = compare_df[compare_df['anomaly']].copy()

if anom_df.empty:
    print('✅ Аномалий не обнаружено! Модель хорошо описывает реальную теплицу.')
else:
    anom_df['timestamp'] = pd.to_datetime(anom_df['timestamp'])
    
    print(f'⚠️  Обнаружено {len(anom_df)} аномальных точек:')
    print(f'   Первая: {anom_df["timestamp"].iloc[0]:%Y-%m-%d %H:%M:%S}')
    print(f'   Последняя: {anom_df["timestamp"].iloc[-1]:%Y-%m-%d %H:%M:%S}')
    print()

    # Группируем непрерывные аномалии в «события»
    anom_df = anom_df.reset_index(drop=True)
    anom_df['gap'] = (anom_df['timestamp'].diff() > pd.Timedelta(minutes=10)).cumsum()
    
    events = []
    for grp_id, grp in anom_df.groupby('gap'):
        duration_min = (grp['timestamp'].iloc[-1] - grp['timestamp'].iloc[0]).seconds // 60
        event = {
            'Начало':      grp['timestamp'].iloc[0].strftime('%H:%M:%S'),
            'Конец':       grp['timestamp'].iloc[-1].strftime('%H:%M:%S'),
            'Длительность': f'{duration_min} мин',
            'Точек':       len(grp),
        }
        # Добавляем максимальное отклонение по каждой переменной
        for delta_col in [c for c in grp.columns if c.startswith('delta_')]:
            var = delta_col.replace('delta_', '')
            max_d = grp[delta_col].abs().max()
            event[f'Δ{var} макс'] = f'{max_d:.2f}'
        events.append(event)

    events_df = pd.DataFrame(events)
    print(f'📋 Аномальные события ({len(events_df)} шт.):')
    print(events_df.to_string(index=False))

    # Диагностика
    print('\n🔍 Возможные причины:')
    if 'delta_tAir' in anom_df.columns:
        mean_d = anom_df['delta_tAir'].mean()
        if mean_d < -2:
            print(f'  🚪 Температура НИЖЕ модели на {abs(mean_d):.1f}°C → возможно открытая дверь / окно')
        elif mean_d > 2:
            print(f'  🔥 Температура ВЫШЕ модели на {mean_d:.1f}°C → проблема с охлаждением / вентиляцией')
    if 'delta_RH' in anom_df.columns:
        mean_rh = anom_df['delta_RH'].mean()
        if mean_rh < -10:
            print(f'  💨 Влажность НИЖЕ модели → повышенная вентиляция или сухой воздух снаружи')
        elif mean_rh > 10:
            print(f'  💧 Влажность ВЫШЕ модели → риск конденсации, недостаточная вентиляция')
    if 'delta_co2' in anom_df.columns:
        mean_co2 = anom_df['delta_co2'].mean()
        if mean_co2 < -100:
            print(f'  🌬 CO₂ НИЖЕ модели → повышенный воздухообмен (открытая дверь/окно?)')
        elif mean_co2 > 100:
            print(f'  🌿 CO₂ ВЫШЕ модели → высокое дыхание / недостаточная вентиляция')

## 🎛 5. Интерактивный просмотр

In [ ]:
out = widgets.Output()

# Переменные для выбора
var_options = []
if 'tAir_sim' in compare_df.columns:
    var_options.append(('🌡 Температура воздуха', 'tAir'))
if 'RH_sim' in compare_df.columns:
    var_options.append(('💧 Влажность', 'RH'))
if 'co2_sim' in compare_df.columns:
    var_options.append(('🌬 CO₂', 'co2'))

# Также добавляем переменные только из симуляции
sim_only_options = [
    ('🌱 Биомасса плодов (cFruit)', 'cFruit_sim'),
    ('🔋 Буфер ассимилятов (cBuf)', 'cBuf_sim'),
    ('🌿 Темп. полога (tCan)', 'tCan_sim'),
    ('🔥 Трубы отопления (tPipe)', 'tPipe_sim'),
]

# Добавляем переменные из симуляции в compare_df
t0_dt = sensor_df['timestamp'].iloc[0]
for col in ['cFruit', 'cBuf', 'tCan', 'tPipe']:
    if col in sim_df.columns:
        sim_ts = pd.to_datetime([t0_dt + __import__('datetime').timedelta(seconds=float(s)) for s in sim_df['Time']])
        sim_series = pd.Series(sim_df[col].values, index=sim_ts)
        compare_df_ts = pd.to_datetime(compare_df['timestamp'])
        compare_df[f'{col}_sim'] = sim_series.reindex(sim_series.index.union(compare_df_ts)).interpolate('time').reindex(compare_df_ts).values

all_options = var_options + [(label, key) for label, key in sim_only_options if key in compare_df.columns]

w_var = widgets.Dropdown(
    options=all_options if all_options else [('Нет данных', None)],
    description='Переменная:',
    layout=Layout(width='400px'),
    style={'description_width': '110px'}
)
w_hours = widgets.IntSlider(
    value=min(HOURS_BACK, 24), min=1, max=HOURS_BACK, step=1,
    description='Показать ч:',
    layout=Layout(width='400px'),
    style={'description_width': '110px'}
)
w_show_anom = widgets.Checkbox(value=True, description='Показать аномалии')

COLORS = {'tAir': '#58a6ff', 'RH': '#79c0ff', 'co2': '#3fb950',
          'cFruit_sim': '#ff7b72', 'cBuf_sim': '#d2a8ff',
          'tCan_sim': '#3fb950', 'tPipe_sim': '#f0883e'}
UNITS  = {'tAir': '°C', 'RH': '%', 'co2': 'ppm',
          'cFruit_sim': 'мг/м²', 'cBuf_sim': 'мг/м²',
          'tCan_sim': '°C', 'tPipe_sim': '°C'}

def plot_var(var_key, show_hours, show_anom):
    with out:
        clear_output(wait=True)
        if var_key is None:
            print('Нет доступных переменных')
            return

        cdf = compare_df.copy()
        cdf['timestamp'] = pd.to_datetime(cdf['timestamp'])
        t_max = cdf['timestamp'].max()
        t_min = t_max - __import__('datetime').timedelta(hours=show_hours)
        cdf = cdf[cdf['timestamp'] >= t_min]

        sim_col  = f'{var_key}_sim' if f'{var_key}_sim' in cdf.columns else var_key
        real_col = f'{var_key}_real' if f'{var_key}_real' in cdf.columns else None
        color    = COLORS.get(var_key, '#58a6ff')
        unit     = UNITS.get(var_key, '?')

        if sim_col not in cdf.columns:
            print(f'⚠️  Переменная {sim_col} не найдена')
            return

        has_real = real_col and real_col in cdf.columns and cdf[real_col].notna().any()
        n_rows   = 2 if has_real else 1
        fig, axes = plt.subplots(n_rows, 1, figsize=(14, 4*n_rows), sharex=True)
        if n_rows == 1:
            axes = [axes]

        label_name = next((lbl for lbl, k in all_options if k == var_key), var_key)
        fig.suptitle(f'📊 {label_name}', fontsize=13)

        # График 1: Модель (+ датчик если есть)
        ax = axes[0]
        ax.plot(cdf['timestamp'], cdf[sim_col], color=color, lw=2, label='🔵 Модель')
        if has_real:
            ax.scatter(cdf['timestamp'], cdf[real_col], color='#ff7b72',
                       s=8, alpha=0.6, label='🔴 Датчик')
        if show_anom and 'anomaly' in cdf.columns:
            anom = cdf['anomaly'].values
            for start, end in _get_anomaly_ranges(cdf['timestamp'].values, anom):
                ax.axvspan(start, end, alpha=0.2, color='#ff7b72')
        ax.set_ylabel(unit)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.35)

        # График 2: Отклонение
        if has_real:
            delta_col = f'delta_{var_key}'
            ax2 = axes[1]
            if delta_col in cdf.columns:
                delta = cdf[delta_col]
                ax2.fill_between(cdf['timestamp'], delta, alpha=0.4, color=color)
                ax2.plot(cdf['timestamp'], delta, color=color, lw=1.2)
                ax2.axhline(0, color='#3fb950', lw=1.5)
                thr = delta.std() * 3
                ax2.axhline(+thr, color='#ff7b72', ls='--', lw=1)
                ax2.axhline(-thr, color='#ff7b72', ls='--', lw=1)
            ax2.set_ylabel(f'Δ {unit}')
            ax2.set_title('Отклонение (Модель − Датчик)')
            ax2.grid(True, alpha=0.35)

        for ax in axes:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m %H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=max(1, show_hours//8)))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

        plt.tight_layout()
        plt.show()

widgets.interactive_output(
    plot_var,
    {'var_key': w_var, 'show_hours': w_hours, 'show_anom': w_show_anom}
)
ui = VBox([
    widgets.HTML('<h4 style="color:#e6edf3">⚙️ Выберите переменную и период</h4>'),
    HBox([w_var, w_hours]),
    w_show_anom,
])
display(ui, out)
if all_options:
    plot_var(all_options[0][1], min(HOURS_BACK, 24), True)

## 📈 6. Сводка и рекомендации

In [ ]:
cdf = compare_df.copy()

print('='*60)
print(f'📋 СВОДКА ЦИФРОВОГО ДВОЙНИКА')
print(f'   Устройство: #{DEVICE_ID}  |  Период: {HOURS_BACK} ч')
print('='*60)

metrics = [
    ('tAir',  '°C',   'Температура воздуха'),
    ('RH',    '%',    'Влажность воздуха'),
    ('co2',   'ppm',  'CO₂'),
]
for var, unit, label in metrics:
    sim_col  = f'{var}_sim'
    real_col = f'{var}_real'
    if sim_col in cdf.columns and real_col in cdf.columns:
        rmse = ((cdf[sim_col] - cdf[real_col])**2).mean()**0.5
        bias = (cdf[sim_col] - cdf[real_col]).mean()
        print(f'  {label:30s}: RMSE={rmse:.2f} {unit}  |  Смещение={bias:+.2f} {unit}')

print()
n_anom = cdf['anomaly'].sum()
pct    = n_anom / len(cdf) * 100
print(f'  Аномалий: {n_anom} / {len(cdf)} точек ({pct:.1f}%)')

print()
print('🔍 Оценка состояния теплицы:')
if pct < 5:
    print('  ✅ НОРМА — модель хорошо описывает реальность')
elif pct < 15:
    print('  ⚠️  ВНИМАНИЕ — есть незначительные отклонения')
    print('     Рекомендуется проверить калибровку датчиков')
else:
    print('  🔴 ТРЕВОГА — значительные расхождения между моделью и датчиками!')
    print('     Возможные причины:')
    print('     • Открытая дверь / форточка')
    print('     • Неисправность системы отопления')
    print('     • Ошибка или дрейф датчика')
    print('     • Нестандартные погодные условия')

print()
print('📡 Состояние реальной теплицы (последнее показание):')
last = sensor_df.iloc[-1]
fields = [
    ('airTemperature',    '°C',   'tAir датчик'),
    ('airHumidity',       '%',    'RH датчик'),
    ('co2Concentration',  'ppm',  'CO₂ датчик'),
    ('outsideTemperature','°C',   'tOut'),
    ('windSpeed',         'м/с',  'Ветер'),
]
for field, unit, label in fields:
    if field in last.index and pd.notna(last[field]):
        print(f'  {label:25s}: {last[field]:.1f} {unit}')

print('='*60)